# AQI Predictor — Lahore | Hourly Feature Pipeline

Keeps the `lahore_aqi_features` feature group current between full backfills. Pulls a
recent window of weather + air quality from Open-Meteo's **live** endpoints (not the ERA5
archive `backfill.ipynb` uses, which lags ~5 days), recomputes the same lag/rolling
features, and inserts only the rows newer than what's already in the store.

Meant to run hourly via `.github/workflows/feature-pipeline.yml` (papermill executes this
notebook headlessly), but you can also just Run All manually any time.

Env vars: `HOPSWORKS_KEY` (required), `HOPSWORKS_PROJECT`, `HOPSWORKS_HOST`.

In [ ]:
import os
from datetime import date

import numpy as np
import pandas as pd
import requests
import hopsworks

In [ ]:
LAHORE = {"latitude": 31.5657, "longitude": 74.3142}
CITY = "Lahore"

AQI_URL = "https://air-quality-api.open-meteo.com/v1/air-quality"
FORECAST_URL = "https://api.open-meteo.com/v1/forecast"

AQI_HOURLY = ("us_aqi,pm2_5,pm10,carbon_monoxide,nitrogen_dioxide,"
              "sulphur_dioxide,ozone,dust,aerosol_optical_depth")
WX_HOURLY = ("temperature_2m,relative_humidity_2m,surface_pressure,"
             "wind_speed_10m,wind_direction_10m,cloud_cover,dew_point_2m,precipitation")

# Enough trailing context to seed the 24h lag/rolling features on the newest row,
# plus a safety margin in case a run gets skipped.
LOOKBACK_DAYS = 4

## 1. Fetch + parse helpers

In [ ]:
def hourly_to_frame(resp):
    df = pd.DataFrame(resp["hourly"])
    df["time"] = pd.to_datetime(df["time"])
    return df


def fetch_recent_window(start_date, end_date):
    """Weather + air quality for [start_date, end_date] from Open-Meteo's live models
    (no ERA5 archive lag), merged on time."""
    wx = hourly_to_frame(requests.get(FORECAST_URL, params={
        **LAHORE, "hourly": WX_HOURLY,
        "start_date": start_date, "end_date": end_date, "timezone": "auto",
    }).json())
    aqi = hourly_to_frame(requests.get(AQI_URL, params={
        **LAHORE, "hourly": AQI_HOURLY,
        "start_date": start_date, "end_date": end_date, "timezone": "auto",
    }).json())
    return wx.merge(aqi, on="time", how="inner")

## 2. Feature engineering

Identical logic to `backfill.ipynb`, kept in sync by hand since this notebook only ever
processes a small recent window rather than the full history.

In [ ]:
def add_time_features(df):
    ts = df["time"]
    df["hour"] = ts.dt.hour
    df["day"] = ts.dt.day
    df["month"] = ts.dt.month
    df["day_of_week"] = ts.dt.dayofweek
    df["is_weekend"] = df["day_of_week"].isin([5, 6]).astype(int)
    df["hour_sin"] = np.sin(2 * np.pi * df["hour"] / 24)
    df["hour_cos"] = np.cos(2 * np.pi * df["hour"] / 24)
    df["month_sin"] = np.sin(2 * np.pi * df["month"] / 12)
    df["month_cos"] = np.cos(2 * np.pi * df["month"] / 12)
    return df


def add_derived_features(df):
    df["pm25_pm10_ratio"] = df["pm2_5"] / df["pm10"]
    df["temp_humidity_index"] = df["temperature_2m"] * df["relative_humidity_2m"] / 100
    df["aqi_category"] = pd.cut(
        df["us_aqi"],
        bins=[-1, 50, 100, 150, 200, 300, 500],
        labels=["Good", "Moderate", "Unhealthy(SG)", "Unhealthy", "Very Unhealthy", "Hazardous"],
    ).astype(str).replace("nan", "Unknown")
    return df


def add_sequential_features(df):
    df = df.sort_values("time").reset_index(drop=True)
    df["aqi_prev"] = df["us_aqi"].shift(1)
    hours_elapsed = df["time"].diff().dt.total_seconds() / 3600
    df["aqi_change_rate"] = (df["us_aqi"] - df["aqi_prev"]) / hours_elapsed
    for lag in (1, 3, 24):
        df[f"aqi_lag_{lag}h"] = df["us_aqi"].shift(lag)
        df[f"pm25_lag_{lag}h"] = df["pm2_5"].shift(lag)
    df["aqi_roll_mean_24h"] = df["us_aqi"].shift(1).rolling(24, min_periods=6).mean()
    df["pm25_roll_mean_24h"] = df["pm2_5"].shift(1).rolling(24, min_periods=6).mean()
    df["aqi_roll_max_24h"] = df["us_aqi"].shift(1).rolling(24, min_periods=6).max()
    return df


def build_features(df):
    df = df.copy()
    df["city"] = CITY
    df = add_time_features(df)
    df = add_derived_features(df)
    df = add_sequential_features(df)
    return df

## 3. Connect, diff against the store, insert only what's new

In [ ]:
project = hopsworks.login(
    project=os.environ.get("HOPSWORKS_PROJECT", "project123456789"),
    host=os.environ.get("HOPSWORKS_HOST", "eu-west.cloud.hopsworks.ai"),
    port=443,
    api_key_value=os.environ["HOPSWORKS_KEY"],
)
fs = project.get_feature_store()
fg = fs.get_feature_group("lahore_aqi_features", version=1)

In [ ]:
existing = fg.read()
# Hopsworks round-trips the naive local timestamps we wrote with a UTC label
# but no actual shift, so drop the label rather than converting.
last_time = pd.to_datetime(existing["time"]).dt.tz_localize(None).max()
print("latest row already in the feature store:", last_time)

In [ ]:
start = (last_time - pd.Timedelta(days=LOOKBACK_DAYS)).date().isoformat()
end = date.today().isoformat()
raw = fetch_recent_window(start, end)
features = build_features(raw)

# the forecast API pads today with hours that haven't happened yet — drop them
now_pkt = pd.Timestamp.utcnow().tz_localize(None) + pd.Timedelta(hours=5)
features = features[
    (features["time"] > last_time) & (features["time"] <= now_pkt)
].reset_index(drop=True)

print("new rows to insert:", len(features))
features.tail()

In [ ]:
if len(features):
    fg.insert(features)
    print(f"Inserted {len(features)} new row(s), up to {features['time'].max()}")
else:
    print("No new rows to insert.")